In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install rasterio shapely -q

In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import xy
import os
from PIL import Image
from shapely.geometry import Point, shape
import json
import gc

tile_paths = [
    '/content/drive/MyDrive/BRACU_DATASET/8k/bracu_full_image4-0.tif',
    '/content/drive/MyDrive/BRACU_DATASET/8k/bracu_full_image4-1.tif',
    '/content/drive/MyDrive/BRACU_DATASET/8k/bracu_full_image4-2.tif',
    '/content/drive/MyDrive/BRACU_DATASET/8k/bracu_full_image4-3.tif',
]
labels_path = '/content/drive/MyDrive/BRACU_DATASET/8k/bracu_labels_roofs4.csv'
output_dir  = '/content/drive/MyDrive/BRACU_DATASET/34k/dataset_34k'
patch_size  = 112

# ==============================
# LOAD LABELS
# ==============================

labels_df = pd.read_csv(labels_path)

def parse_geometry(geo_str):
    try:
        return shape(json.loads(geo_str))
    except:
        return None

labels_df['geometry'] = labels_df['.geo'].apply(parse_geometry)
labels_df = labels_df[labels_df['geometry'].notna()]
print(f"Valid grid cells: {len(labels_df)}")

# ==============================
# FIND LAST PATCH NUMBER
# ==============================

existing = [f for f in os.listdir(output_dir) if f.endswith('.png')]
if existing:
    last_num = max([int(f.replace('patch_','').replace('.png','')) for f in existing])
    count = last_num + 1
else:
    count = 0

print(f"Resuming from patch_{count}")

# ==============================
# MATCH PATCH TO LABEL BY GPS
# ==============================

def get_poverty_score(center_lon, center_lat, labels_df):
    point = Point(center_lon, center_lat)
    for _, row in labels_df.iterrows():
        if row['geometry'].contains(point):
            return row['poverty_score']
    labels_df['dist'] = labels_df['geometry'].apply(
        lambda g: point.distance(g.centroid)
    )
    nearest = labels_df.loc[labels_df['dist'].idxmin()]
    return nearest['poverty_score']

# ==============================
# LOAD EXISTING CSV
# ==============================

existing_csv = os.path.join(output_dir, 'dataset.csv')
if os.path.exists(existing_csv):
    existing_df = pd.read_csv(existing_csv)
    dataset = existing_df.to_dict('records')
else:
    dataset = []

print(f"Existing records in CSV: {len(dataset)}")

# ==============================
# RESUME PATCHING
# ==============================

skipped = 0

for tile_path in tile_paths:
    if count >= 34000:
        break

    print(f"Processing {os.path.basename(tile_path)}...")

    with rasterio.open(tile_path) as src:
        tile_img = src.read()
        tile_transform = src.transform

        tile_img = tile_img.astype(np.float32)
        tile_img = np.nan_to_num(tile_img, nan=0.0, posinf=0.0, neginf=0.0)
        tile_img = np.moveaxis(tile_img, 0, -1)

        for i in range(tile_img.shape[2]):
            band = tile_img[:,:,i]
            bmin, bmax = np.percentile(band[band > 0], [2, 98]) if np.any(band > 0) else (0, 1)
            tile_img[:,:,i] = np.clip((band - bmin) / (bmax - bmin + 1e-8), 0, 1)

        tile_img = (tile_img * 255).astype(np.uint8)
        H, W, _ = tile_img.shape
        print(f"Tile size: {H} x {W}")

        for row in range(0, H - patch_size, patch_size):
            for col in range(0, W - patch_size, patch_size):
                patch = tile_img[row:row+patch_size, col:col+patch_size]

                if patch.mean() < 5:
                    skipped += 1
                    continue

                # Skip already saved patches
                if skipped + count <= 8000:
                    skipped += 1
                    continue

                center_col = col + patch_size // 2
                center_row = row + patch_size // 2
                center_lon, center_lat = xy(tile_transform, center_row, center_col)

                poverty_score = get_poverty_score(center_lon, center_lat, labels_df)

                patch_rgb = patch[:, :, :3]
                pil_img   = Image.fromarray(patch_rgb)
                png_name  = f'patch_{count}.png'
                pil_img.save(os.path.join(output_dir, png_name))

                dataset.append({
                    'image':         png_name,
                    'row':           row,
                    'col':           col,
                    'center_lat':    center_lat,
                    'center_lon':    center_lon,
                    'poverty_score': poverty_score
                })

                count += 1
                if count % 100 == 0:
                    # Save CSV checkpoint every 100 patches
                    pd.DataFrame(dataset).to_csv(existing_csv, index=False)
                    print(f"Patches saved: {count}")

                if count >= 34000:
                    break
            if count >= 34000:
                break

    del tile_img
    gc.collect()
    print(f"Total patches so far: {count}")

# ==============================
# SAVE FINAL CSV
# ==============================

final_df = pd.DataFrame(dataset)
final_df.to_csv(existing_csv, index=False)
print(f"Done! Total patches: {len(final_df)}")
print(final_df.head())

Valid grid cells: 19453
Resuming from patch_8000
Existing records in CSV: 8000
Processing bracu_full_image4-0.tif...
Tile size: 1523 x 2554
Total patches so far: 8003
Processing bracu_full_image4-1.tif...
Tile size: 1523 x 10496
Patches saved: 8100
Patches saved: 8200
Patches saved: 8300
Patches saved: 8400
Patches saved: 8500
Patches saved: 8600
Patches saved: 8700
Total patches so far: 8724
Processing bracu_full_image4-2.tif...
Tile size: 10496 x 2554
Patches saved: 8800
Patches saved: 8900
Patches saved: 9000
Patches saved: 9100
Patches saved: 9200
Patches saved: 9300
Patches saved: 9400
Patches saved: 9500
Patches saved: 9600
Patches saved: 9700
Patches saved: 9800
Patches saved: 9900
Patches saved: 10000
Total patches so far: 10084
Processing bracu_full_image4-3.tif...
Tile size: 10496 x 10496
Patches saved: 10100
Patches saved: 10200
Patches saved: 10300
Patches saved: 10400
Patches saved: 10500
Patches saved: 10600
Patches saved: 10700
Patches saved: 10800
Patches saved: 10900
P